In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [23]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [24]:
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)  

In [25]:
#Encoding categorical data
labelencoder_gender = LabelEncoder()
data['Gender'] = labelencoder_gender.fit_transform(data['Gender'])

In [26]:
onehotencoder_geo = OneHotEncoder()
geo_encoded = onehotencoder_geo.fit_transform(data[['Geography']]).toarray()
geo_df = pd.DataFrame(geo_encoded, columns=onehotencoder_geo.get_feature_names_out(['Geography']))
geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [27]:
data =pd.concat([data.drop('Geography',axis=1),geo_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [28]:
#Splitting the dataset into the Training set and Test set
X = data.drop('EstimatedSalary',axis=1)
y = data['EstimatedSalary']

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

In [31]:
#save the encoders and scalar for later use
import os
os.makedirs('preprocessing_files_regression', exist_ok=True)

with open('preprocessing_files_regression/labelencoder_gender.pkl', 'wb') as file:
        pickle.dump(labelencoder_gender, file)

with open('preprocessing_files_regression/onehotencoder_geo.pkl', 'wb') as file:
        pickle.dump(onehotencoder_geo, file)

with open('preprocessing_files_regression/scalar.pkl', 'wb') as file:
        pickle.dump(scalar, file)

### ANN Regression Problem statement


In [32]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [33]:
model_regression = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

c:\Users\Win\Data Science Course\Udemy NLP and Deep Learning\Project - ANN Classification\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
model_regression.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
model_regression.compile(optimizer='adam', loss='mean_absolute_error',metrics=['mae'])

In [36]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [37]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [38]:
#training the model
history = model_regression.fit(X_train, y_train,
                                epochs=100, 
                                validation_split=0.2,
                                validation_data=(X_test, y_test), 
                                callbacks=[early_stopping, tensorboard_callback])

Epoch 1/100


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100380.3516 - mae: 100380.3516 - val_loss: 98523.2812 - val_mae: 98523.2812
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 99630.4062 - mae: 99630.4062 - val_loss: 96991.9922 - val_mae: 96991.9922
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 96951.8906 - mae: 96951.8906 - val_loss: 93049.5547 - val_mae: 93049.5547
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 91679.7891 - mae: 91679.7891 - val_loss: 86476.2578 - val_mae: 86476.2578
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 83966.4531 - mae: 83966.4531 - val_loss: 77931.6953 - val_mae: 77931.6953
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 74892.1250 - mae: 74892.1250 - val_loss: 69078.9453 - val_mae: 69078.9453
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 66131.1016 - mae: 66131.1016 - val_loss: 61349.0156 - val_mae: 61349.0156
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5905

In [39]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [40]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 20328), started 0:30:23 ago. (Use '!kill 20328' to kill it.)

In [41]:
#Evaluate the model on the test set
test_loss, test_mae = model_regression.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}, Test MAE: {test_mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 50297.7539 - mae: 50297.7539  
Test Loss: 50297.75390625, Test MAE: 50297.75390625


In [42]:
model_regression.save('regression_model.keras')